# Logistic Regression — Beginner-Friendly Explanation

This notebook explains how we separate input features from the target (`closed`), train logistic regression, predict labels, and calculate accuracy.

**Expected files in the same folder:** `crime_train.csv` and `crime_test.csv`.

The code assumes the target column is named `closed` and contains `Yes`/`No`.


## 1. Import libraries and load data

- `pandas` works with tables.
- `numpy` performs numerical calculations.


In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("crime_train.csv")
test = pd.read_csv("crime_test.csv")

print(train.head())


   Unnamed: 0    Num        case_filed       city  area    crime_description  \
0       19794  19795  04-04-2022 18:00     Mumbai   307     VEHICLE - STOLEN   
1       13762  13763  07-27-2021 10:00      Surat   504              ROBBERY   
2       30311  30312  06-16-2023 23:00      Thane   493              ASSAULT   
3        5043   5044  07-29-2020 03:00  Ahmedabad   169  PUBLIC INTOXICATION   
4       35008  35009  12-29-2023 16:00     Mumbai   133  PUBLIC INTOXICATION   

    age sex      weapon         domain  police_department closed  
0  63.0   X      Poison  Violent Crime               12.0     No  
1  57.0   F  Explosives  Violent Crime                3.0     No  
2  33.0   F     Firearm  Violent Crime               18.0     No  
3  76.0   F       Knife    Other Crime                3.0     No  
4  13.0   M         NaN    Other Crime               13.0     No  


## 2. Create `y` and `yt` — the actual answers

```python
y = (train["closed"] == "Yes").astype(int).to_numpy()
yt = (test["closed"] == "Yes").astype(int).to_numpy()
```

For values `Yes, No, Yes, No`:
1. `== "Yes"` produces `True, False, True, False`.
2. `.astype(int)` converts these to `1, 0, 1, 0`.
3. `.to_numpy()` makes a NumPy array.

- `y` = actual training answers.
- `yt` = actual test answers.
- `1` means Yes; `0` means No.


In [2]:
y = (train["closed"] == "Yes").astype(int).to_numpy()
yt = (test["closed"] == "Yes").astype(int).to_numpy()

print("First training answers:", y[:10])
print("First test answers:", yt[:10])


First training answers: [0 0 0 0 0 0 0 0 0 0]
First test answers: [1 1 1 1 1 0 0 1 0 1]


## 3. Create `X` and `Xt` — the input features

`drop()` removes columns from the table.

- `closed` is removed because it is the answer the model must predict.
- `Num` and `Unnamed: 0` are assumed to be identifiers/index columns.
- `case_filed` is excluded in this example.

**Check your CSV:** all listed columns must exist for this exact code to run. If a column is absent, remove its name from the list.

- `X` = training input features.
- `Xt` = test input features.


In [3]:
X = train.drop(columns=["closed", "Num", "Unnamed: 0", "case_filed"])
Xt = test.drop(columns=["closed", "Num", "Unnamed: 0", "case_filed"])

print("Training input shape:", X.shape)
print("Test input shape:", Xt.shape)
print(X.head())


Training input shape: (22489, 8)
Test input shape: (9639, 8)
        city  area    crime_description   age sex      weapon         domain  \
0     Mumbai   307     VEHICLE - STOLEN  63.0   X      Poison  Violent Crime   
1      Surat   504              ROBBERY  57.0   F  Explosives  Violent Crime   
2      Thane   493              ASSAULT  33.0   F     Firearm  Violent Crime   
3  Ahmedabad   169  PUBLIC INTOXICATION  76.0   F       Knife    Other Crime   
4     Mumbai   133  PUBLIC INTOXICATION  13.0   M         NaN    Other Crime   

   police_department  
0               12.0  
1                3.0  
2               18.0  
3                3.0  
4               13.0  


## 4. Prepare the data

We fill missing values, clip extreme numeric values, convert text categories to numeric columns, and scale the features. Values used for preprocessing are learned from training data and applied to test data.


In [5]:
# Fill missing values using training-data mean/mode
for c in X.columns:
    if pd.api.types.is_numeric_dtype(X[c]):
        value = X[c].mean()
    else:
        mode = X[c].mode()
        value = mode.iloc[0] if len(mode) > 0 else "Unknown"

    X[c] = X[c].fillna(value)
    Xt[c] = Xt[c].fillna(value)

# Clip outliers in numeric columns
num_cols = X.select_dtypes(include=np.number).columns.tolist()

q1 = X[num_cols].quantile(0.25)
q3 = X[num_cols].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

X[num_cols] = X[num_cols].clip(lower=low, upper=high, axis=1)
Xt[num_cols] = Xt[num_cols].clip(lower=low, upper=high, axis=1)

# Convert text categories into numeric dummy columns
X = pd.get_dummies(X, dtype=int)
Xt = pd.get_dummies(Xt, dtype=int)

# Ensure test columns match training columns
Xt = Xt.reindex(columns=X.columns, fill_value=0)

# Scale using training-data statistics
mean = X.mean()
rng = (X.max() - X.min()).replace(0, 1)

X = ((X - mean) / rng).to_numpy(dtype=float)
Xt = ((Xt - mean) / rng).to_numpy(dtype=float)

print("Prepared training shape:", X.shape)
print("Prepared test shape:", Xt.shape)


Prepared training shape: (22489, 66)
Prepared test shape: (9639, 66)


## 5. Train logistic regression with gradient descent

- `w` = weights learned by the model.
- `b` = bias.
- `alpha` = learning rate, controlling the size of each update.
- `A` = predicted probabilities during training.
- `cost` = log loss; training aims to reduce it.


In [6]:
m, n = X.shape
w = np.zeros(n)
b = 0.0
alpha = 0.1

for i in range(2000):
    z = np.dot(X, w) + b
    z = np.clip(z, -500, 500)
    A = 1 / (1 + np.exp(-z))

    eps = 1e-9
    cost = -np.mean(
        y * np.log(A + eps) +
        (1 - y) * np.log(1 - A + eps)
    )

    error = A - y
    dw = np.dot(X.T, error) / m
    db = np.sum(error) / m

    w = w - alpha * dw
    b = b - alpha * db

    # Reduce learning rate by 10% every 200 iterations
    if (i + 1) % 200 == 0:
        alpha = alpha * 0.9
        print("Iteration:", i + 1,
              "Log Loss:", cost,
              "Learning Rate:", alpha)


Iteration: 200 Log Loss: 0.6927066164926355 Learning Rate: 0.09000000000000001
Iteration: 400 Log Loss: 0.6925529300320267 Learning Rate: 0.08100000000000002
Iteration: 600 Log Loss: 0.6924785554284283 Learning Rate: 0.07290000000000002
Iteration: 800 Log Loss: 0.6924355720846965 Learning Rate: 0.06561000000000002
Iteration: 1000 Log Loss: 0.6924078734423474 Learning Rate: 0.05904900000000002
Iteration: 1200 Log Loss: 0.6923887196423835 Learning Rate: 0.05314410000000002
Iteration: 1400 Log Loss: 0.6923748089146308 Learning Rate: 0.04782969000000002
Iteration: 1600 Log Loss: 0.6923643329304671 Learning Rate: 0.043046721000000024
Iteration: 1800 Log Loss: 0.6923562191877847 Learning Rate: 0.03874204890000002
Iteration: 2000 Log Loss: 0.6923497929055978 Learning Rate: 0.03486784401000002


## 6. Predict on test data

1. Calculate a score using test inputs, weights, and bias.
2. Sigmoid converts each score into a probability.
3. If probability is at least `0.5`, predict `1` (Yes); otherwise predict `0` (No).


In [7]:
z = np.dot(Xt, w) + b
z = np.clip(z, -500, 500)
prob = 1 / (1 + np.exp(-z))

pred_test = (prob >= 0.5).astype(int)

print("First probabilities:", prob[:10])
print("First predictions:", pred_test[:10])


First probabilities: [0.47548788 0.51074491 0.47845847 0.5360635  0.53151052 0.51564619
 0.48653513 0.4999756  0.48486729 0.5205798 ]
First predictions: [0 1 0 1 1 1 0 0 0 1]


## 7. Calculate accuracy

`pred_test == yt` checks whether each prediction matches the actual test answer. `np.mean()` calculates the fraction correct; multiplying by 100 converts it to a percentage.


In [8]:
accuracy = np.mean(pred_test == yt) * 100
print("Accuracy:", accuracy, "%")


Accuracy: 49.486461251167135 %


## 8. Save predictions

The output file contains the actual label, predicted label, and predicted probability for each test row.


In [9]:
result = pd.DataFrame({
    "actual_closed": test["closed"],
    "predicted_closed": np.where(pred_test == 1, "Yes", "No"),
    "probability_closed": prob
})

result.to_csv("crime_predictions.csv", index=False)
print("Predictions saved to crime_predictions.csv")
print(result.head(10))


Predictions saved to crime_predictions.csv
  actual_closed predicted_closed  probability_closed
0           Yes               No            0.475488
1           Yes              Yes            0.510745
2           Yes               No            0.478458
3           Yes              Yes            0.536064
4           Yes              Yes            0.531511
5            No              Yes            0.515646
6            No               No            0.486535
7           Yes               No            0.499976
8            No               No            0.484867
9           Yes              Yes            0.520580
